In [54]:
from pyspark.sql import functions as F
import time
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
from src.spark_session import get_spark
from src.config import ORDER_ITEMS_SILVER_PATH, ORDER_ITEMS_WITH_PRODUCTS_PATH, PRODUCTS_SILVER_PATH

In [2]:
spark = get_spark("DeltaBronze")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/07 14:38:39 WARN Utils: Your hostname, Branimirs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.199 instead (on interface en0)
26/08/07 14:38:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/07 14:38:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applic

In [3]:
orders_items_with_products  = spark.read.parquet(str(ORDER_ITEMS_WITH_PRODUCTS_PATH))

orders_items_with_products .printSchema()
print("Rows: ", orders_items_with_products .count())
print("Partitions: ", orders_items_with_products .rdd.getNumPartitions())

root
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- item_total: decimal(13,2) (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = tr

In [4]:
expensive_items = (
    orders_items_with_products
    .filter(F.col("price") > 100)
    .select("order_id", "order_item_id", "product_id", "price")
)

expensive_items.explain("formatted")

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [4]: [product_id#0, order_id#1, order_item_id#2, price#5]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items_with_products]
PushedFilters: [IsNotNull(price), GreaterThan(price,100.00)]
ReadSchema: struct<product_id:string,order_id:string,order_item_id:int,price:decimal(12,2)>

(2) ColumnarToRow [codegen id : 1]
Input [4]: [product_id#0, order_id#1, order_item_id#2, price#5]

(3) Filter [codegen id : 1]
Input [4]: [product_id#0, order_id#1, order_item_id#2, price#5]
Condition : (isnotnull(price#5) AND (price#5 > 100.00))

(4) Project [codegen id : 1]
Output [4]: [order_id#1, order_item_id#2, product_id#0, price#5]
Input [4]: [product_id#0, order_id#1, order_item_id#2, price#5]




In [5]:
revenue_by_category = (
    orders_items_with_products
    .groupBy("product_category_name")
    .agg(
        F.sum("price").alias("revenue"),
        F.countDistinct("order_id").alias("orders"),
    )
)

revenue_by_category.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- HashAggregate (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- HashAggregate (4)
            +- Exchange (3)
               +- HashAggregate (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [order_id#1, price#5, product_category_name#16]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items_with_products]
ReadSchema: struct<order_id:string,price:decimal(12,2),product_category_name:string>

(2) HashAggregate
Input [3]: [order_id#1, price#5, product_category_name#16]
Keys [2]: [product_category_name#16, order_id#1]
Functions [1]: [partial_sum(price#5)]
Aggregate Attributes [1]: [sum(price#5)#70]
Results [4]: [product_category_name#16, order_id#1, sum#74, isEmpty#75]

(3) Exchange
Input [4]: [product_category_name#16, order_id#1, sum#74, isEmpty#75]
Arguments: hashpartitioning(product_category_name#16, order_id#1, 200), ENSUR

In [6]:
simple_df = (
    orders_items_with_products
    .filter(F.col("price") > 100)
    .select("order_id", "product_id", "price")
)

simple_df.explain("formatted")

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [product_id#0, order_id#1, price#5]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items_with_products]
PushedFilters: [IsNotNull(price), GreaterThan(price,100.00)]
ReadSchema: struct<product_id:string,order_id:string,price:decimal(12,2)>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [product_id#0, order_id#1, price#5]

(3) Filter [codegen id : 1]
Input [3]: [product_id#0, order_id#1, price#5]
Condition : (isnotnull(price#5) AND (price#5 > 100.00))

(4) Project [codegen id : 1]
Output [3]: [order_id#1, product_id#0, price#5]
Input [3]: [product_id#0, order_id#1, price#5]




In [7]:
grouped_df = (
    orders_items_with_products
    .groupBy("product_category_name")
    .agg(
        F.sum("price").alias("revenue"),
    )
)

grouped_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [price#5, product_category_name#16]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items_with_products]
ReadSchema: struct<price:decimal(12,2),product_category_name:string>

(2) HashAggregate
Input [2]: [price#5, product_category_name#16]
Keys [1]: [product_category_name#16]
Functions [1]: [partial_sum(price#5)]
Aggregate Attributes [2]: [sum#103, isEmpty#104]
Results [3]: [product_category_name#16, sum#105, isEmpty#106]

(3) Exchange
Input [3]: [product_category_name#16, sum#105, isEmpty#106]
Arguments: hashpartitioning(product_category_name#16, 200), ENSURE_REQUIREMENTS, [plan_id=107]

(4) HashAggregate
Input [3]: [product_category_name#16, sum#105, isEmpty#106]
Keys [1]: [product_category_name#16]
Functions [1]: [sum(price#5)]
Aggreg

In [8]:
sorted_df = orders_items_with_products.orderBy(F.col("price").desc())
sorted_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (4)
+- Sort (3)
   +- Exchange (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [21]: [product_id#0, order_id#1, order_item_id#2, seller_id#3, shipping_limit_date#4, price#5, freight_value#6, item_total#7, customer_id#8, customer_unique_id#9, customer_state#10, order_status#11, order_purchase_timestamp#12, delivery_days#13, delivery_delay_days#14, is_late#15, product_category_name#16, product_weight_g#17, product_length_cm#18, product_height_cm#19, product_width_cm#20]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items_with_products]
ReadSchema: struct<product_id:string,order_id:string,order_item_id:int,seller_id:string,shipping_limit_date:timestamp,price:decimal(12,2),freight_value:decimal(12,2),item_total:decimal(13,2),customer_id:string,customer_unique_id:string,customer_state:string,order_status:string,order_purchase_timestamp:timestamp,delivery_days:i

In [9]:
order_items = spark.read.parquet(str(ORDER_ITEMS_SILVER_PATH))
products = spark.read.parquet(str(PRODUCTS_SILVER_PATH))

In [10]:
order_items.printSchema()
products.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- item_total: decimal(13,2) (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [11]:
joined_df = (
    order_items
    .join(products, on="product_id", how="left")
)

joined_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- Project (6)
   +- BroadcastHashJoin LeftOuter BuildRight (5)
      :- Scan parquet  (1)
      +- BroadcastExchange (4)
         +- Filter (3)
            +- Scan parquet  (2)


(1) Scan parquet 
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:decimal(12,2),freight_value:decimal(12,2),item_total:decimal(13,2)>

(2) Scan parquet 
Output [9]: [product_id#115, product_category_name#116, product_name_length#117, product_description_length#118, product_photos_qty#119, product_weight_g#120, product_length_cm#121, product_height_cm#122, product_width_cm#123]
Batched: true
Location: InMemoryFileI

In [12]:
print(
    "order_items partitions:",
    order_items.rdd.getNumPartitions()
)

print(
    "products partitions:",
    products.rdd.getNumPartitions()
)

print(
    "joined dataframe partitions:",
    joined_df.rdd.getNumPartitions()
)

order_items partitions: 2
products partitions: 2
joined dataframe partitions: 2


In [13]:
repartitioned = order_items.repartition(4)

print(
    "After repartition:",
    repartitioned.rdd.getNumPartitions()
)

repartitioned.explain("formatted")

After repartition: 4
== Physical Plan ==
AdaptiveSparkPlan (7)
+- == Final Plan ==
   ResultQueryStage (5)
   +- ShuffleQueryStage (4), Statistics(sizeInBytes=18.0 MiB, rowCount=1.13E+5)
      +- Exchange (3)
         +- * ColumnarToRow (2)
            +- Scan parquet  (1)
+- == Initial Plan ==
   Exchange (6)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:decimal(12,2),freight_value:decimal(12,2),item_total:decimal(13,2)>

(2) ColumnarToRow [codegen id : 1]
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, 

In [14]:
coalesced = repartitioned.coalesce(2)

print(
    "After coalesce:",
    coalesced.rdd.getNumPartitions()
)

coalesced.explain("formatted")

After coalesce: 2
== Physical Plan ==
AdaptiveSparkPlan (9)
+- == Final Plan ==
   ResultQueryStage (6)
   +- Coalesce (5)
      +- ShuffleQueryStage (4), Statistics(sizeInBytes=18.0 MiB, rowCount=1.13E+5)
         +- Exchange (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)
+- == Initial Plan ==
   Coalesce (8)
   +- Exchange (7)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:decimal(12,2),freight_value:decimal(12,2),item_total:decimal(13,2)>

(2) ColumnarToRow [codegen id : 1]
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipp

In [15]:
repartition_by_product = order_items.repartition(
    4,
    "product_id"
)

repartition_by_product.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (3)
+- Exchange (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:decimal(12,2),freight_value:decimal(12,2),item_total:decimal(13,2)>

(2) Exchange
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Arguments: hashpartitioning(product_id#109, 4), REPARTITION_BY_NUM, [plan_id=259]

(3) AdaptiveSparkPlan
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Argumen

In [16]:
direct_coalesce = order_items.coalesce(1)

direct_coalesce.explain("formatted")

== Physical Plan ==
Coalesce (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:decimal(12,2),freight_value:decimal(12,2),item_total:decimal(13,2)>

(2) ColumnarToRow [codegen id : 1]
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]

(3) Coalesce
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Arguments: 1




In [17]:
direct_repartition = order_items.repartition(1)

direct_repartition.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (3)
+- Exchange (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:decimal(12,2),freight_value:decimal(12,2),item_total:decimal(13,2)>

(2) Exchange
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Arguments: SinglePartition, REPARTITION_BY_NUM, [plan_id=280]

(3) AdaptiveSparkPlan
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Arguments: isFinalPlan=fals

In [18]:
spark.conf.get("spark.sql.shuffle.partitions")

'200'

In [19]:
direct_coalesce = order_items.coalesce(1)
direct_coalesce.explain("formatted")

== Physical Plan ==
Coalesce (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:decimal(12,2),freight_value:decimal(12,2),item_total:decimal(13,2)>

(2) ColumnarToRow [codegen id : 1]
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]

(3) Coalesce
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Arguments: 1




In [20]:
direct_repartition = order_items.repartition(1)
direct_repartition.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (3)
+- Exchange (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:decimal(12,2),freight_value:decimal(12,2),item_total:decimal(13,2)>

(2) Exchange
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Arguments: SinglePartition, REPARTITION_BY_NUM, [plan_id=301]

(3) AdaptiveSparkPlan
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Arguments: isFinalPlan=fals

In [21]:
print(
    "shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

shuffle partitions: 200


In [22]:
print(
    "order_items partitions:",
    order_items.rdd.getNumPartitions()
)

order_items partitions: 2


In [25]:
partition_counts = (
    order_items
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("row_count"),
    )
    .orderBy("partition_id")
)

print(partition_counts.show())

+------------+---------+
|partition_id|row_count|
+------------+---------+
|           0|    60876|
|           1|    51774|
+------------+---------+

None


In [28]:
balanced_df = order_items.repartition(4)

balanced_partition_counts = (
    balanced_df
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("row_count"),
    )
    .orderBy("partition_id")
)

print(balanced_partition_counts.show())

+------------+---------+
|partition_id|row_count|
+------------+---------+
|           0|    28162|
|           1|    28163|
|           2|    28163|
|           3|    28162|
+------------+---------+

None


In [31]:
repartition_by_product = order_items.repartition(4, "product_id")

product_partition_counts = (
    repartition_by_product
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("row_count"),
    )
    .orderBy("partition_id")
)

print(product_partition_counts.show())

+------------+---------+
|partition_id|row_count|
+------------+---------+
|           0|    30324|
|           1|    27546|
|           2|    25960|
|           3|    28820|
+------------+---------+

None


In [32]:
state_distribution = (
    orders_items_with_products
    .groupBy("customer_state")
    .agg(
        F.count("*").alias("row_count"),
    )
    .orderBy(F.col("row_count").desc())
)

state_distribution.show(30, truncate=False)

+--------------+---------+
|customer_state|row_count|
+--------------+---------+
|sp            |47449    |
|rj            |14579    |
|mg            |13129    |
|rs            |6235     |
|pr            |5740     |
|sc            |4176     |
|ba            |3799     |
|df            |2406     |
|go            |2333     |
|es            |2256     |
|pe            |1806     |
|ce            |1478     |
|pa            |1080     |
|mt            |1055     |
|ma            |824      |
|ms            |819      |
|pb            |602      |
|pi            |542      |
|rn            |529      |
|al            |444      |
|se            |385      |
|to            |315      |
|ro            |278      |
|am            |165      |
|ac            |92       |
|ap            |82       |
|rr            |52       |
+--------------+---------+



In [33]:
category_distribution = (
    orders_items_with_products
    .groupBy("product_category_name")
    .agg(
        F.count("*").alias("row_count"),
    )
    .orderBy(F.col("row_count").desc())
)

category_distribution.show(20, truncate=False)

+---------------------------+---------+
|product_category_name      |row_count|
+---------------------------+---------+
|cama_mesa_banho            |11115    |
|beleza_saude               |9670     |
|esporte_lazer              |8641     |
|moveis_decoracao           |8334     |
|informatica_acessorios     |7827     |
|utilidades_domesticas      |6964     |
|relogios_presentes         |5991     |
|telefonia                  |4545     |
|ferramentas_jardim         |4347     |
|automotivo                 |4235     |
|brinquedos                 |4117     |
|cool_stuff                 |3796     |
|perfumaria                 |3419     |
|bebes                      |3065     |
|eletronicos                |2767     |
|papelaria                  |2517     |
|fashion_bolsas_e_acessorios|2031     |
|pet_shop                   |1947     |
|moveis_escritorio          |1691     |
|unknown                    |1603     |
+---------------------------+---------+
only showing top 20 rows


In [34]:
product_distribution = (
    order_items
    .groupBy("product_id")
    .agg(
        F.count("*").alias("row_count"),
    )
    .orderBy(F.col("row_count").desc())
)

product_distribution.show(20, truncate=False)

+--------------------------------+---------+
|product_id                      |row_count|
+--------------------------------+---------+
|aca2eb7d00ea1a7b8ebd4e68314663af|527      |
|99a4788cb24856965c36a24e339b6058|488      |
|422879e10f46682990de24d770e7f83d|484      |
|389d119b48cf3043d311335e499d9c6b|392      |
|368c6c730842d78016ad823897a372db|388      |
|53759a2ecddad2bb87a079a1f1519f73|373      |
|d1c427060a0f73f6b889a5c7c61f2ac4|343      |
|53b36df67ebb7c41585e8d54d6772e08|323      |
|154e7e31ebfa092203795c972e5804a6|281      |
|3dd2a17168ec895c781a9191c1e95ad7|274      |
|2b4609f8948be18874494203496bc318|260      |
|7c1bd920dbdf22470b68bde975dd3ccf|231      |
|a62e25e09e05e6faf31d90c6ec1aa3d1|226      |
|5a848e4ab52fd5445cdc07aab1c40e48|197      |
|bb50f2e236e5eea0100680137654686c|195      |
|e0d64dcfaa3b6db5c54ca298ae101d05|194      |
|e53e557d5a159f5aa2c5e995dfdf244b|183      |
|42a2c92a0979a949ca4ea89ec5c7b934|183      |
|b532349fe46b38fbc7bb3914c1bdae07|169      |
|35afc9736

In [35]:
skewed_data = (
    [("popular", i) for i in range(9000)]
    +
    [("rare_a", i) for i in range(500)]
    +
    [("rare_b", i) for i in range(500)]
)

skewed_df = spark.createDataFrame(
    skewed_data,
    ["category", "value"]
)

In [37]:
skewed_repartitioned = skewed_df.repartition(
    4,
    "category"
)

In [40]:
(
    skewed_repartitioned
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("row_count")
    )
    .orderBy("partition_id")
    .show()
)

+------------+---------+
|partition_id|row_count|
+------------+---------+
|           2|      500|
|           3|     9500|
+------------+---------+



In [39]:
skewed_balanced = skewed_df.repartition(4)

(
    skewed_balanced
    .withColumn(
        "partition_id",
        F.spark_partition_id()
    )
    .groupBy("partition_id")
    .agg(
        F.count("*").alias("row_count")
    )
    .orderBy("partition_id")
    .show()
)

+------------+---------+
|partition_id|row_count|
+------------+---------+
|           0|     2500|
|           1|     2500|
|           2|     2500|
|           3|     2500|
+------------+---------+



In [41]:
order_items.rdd.getNumPartitions()

2

In [42]:
balanced_partition_counts.show()

+------------+---------+
|partition_id|row_count|
+------------+---------+
|           0|    28162|
|           1|    28163|
|           2|    28163|
|           3|    28162|
+------------+---------+



In [43]:
product_partition_counts.show()

+------------+---------+
|partition_id|row_count|
+------------+---------+
|           0|    30324|
|           1|    27546|
|           2|    25960|
|           3|    28820|
+------------+---------+



In [44]:
state_distribution.show(30, truncate=False)

+--------------+---------+
|customer_state|row_count|
+--------------+---------+
|sp            |47449    |
|rj            |14579    |
|mg            |13129    |
|rs            |6235     |
|pr            |5740     |
|sc            |4176     |
|ba            |3799     |
|df            |2406     |
|go            |2333     |
|es            |2256     |
|pe            |1806     |
|ce            |1478     |
|pa            |1080     |
|mt            |1055     |
|ma            |824      |
|ms            |819      |
|pb            |602      |
|pi            |542      |
|rn            |529      |
|al            |444      |
|se            |385      |
|to            |315      |
|ro            |278      |
|am            |165      |
|ac            |92       |
|ap            |82       |
|rr            |52       |
+--------------+---------+



In [46]:
delivered_items = (
    orders_items_with_products
    .filter(F.col("order_status") == "delivered")
    .select(
        "order_id",
        "product_id",
        "product_category_name",
        "price",
        "customer_state"
    )
)

print(delivered_items.storageLevel)

Serialized 1x Replicated


In [47]:
delivered_items.cache()

print(delivered_items.storageLevel)

Disk Memory Deserialized 1x Replicated


In [48]:
delivered_items.count()

110197

In [49]:
cached_category_revenue = (
    delivered_items
    .groupBy("product_category_name")
    .agg(
        F.sum("price").alias("revenue")
    )
)

cached_category_revenue.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (10)
+- HashAggregate (9)
   +- Exchange (8)
      +- HashAggregate (7)
         +- InMemoryTableScan (1)
               +- InMemoryRelation (2)
                     +- * Project (6)
                        +- * Filter (5)
                           +- * ColumnarToRow (4)
                              +- Scan parquet  (3)


(1) InMemoryTableScan
Output [2]: [product_category_name#16, price#5]
Arguments: [product_category_name#16, price#5]

(2) InMemoryRelation
Arguments: [order_id#1, product_id#0, product_category_name#16, price#5, customer_state#10], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan parquet 
Output [6]: [product_id#0, order_id#1, price#5, customer_state#10, order_status#11, product_category_name#16]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items_with_products]
PushedFilters: [IsNotNull(order_status), EqualTo(order_status,delivered

In [51]:
uncached_df = (
    orders_items_with_products
    .filter(F.col("order_status") == "delivered")
    .groupBy("product_category_name")
    .agg(
        F.sum("price").alias("revenue"),
        F.countDistinct("order_id").alias("orders")
    )
)

start = time.perf_counter()
uncached_df.count()
print("First action:", time.perf_counter() - start)

start = time.perf_counter()
uncached_df.count()
print("Second action:", time.perf_counter() - start)

First action: 0.4316209999960847
Second action: 0.27377920801518485


In [52]:
cached_df = (
    orders_items_with_products
    .filter(F.col("order_status") == "delivered")
    .groupBy("product_category_name")
    .agg(
        F.sum("price").alias("revenue"),
        F.countDistinct("order_id").alias("orders")
    )
    .cache()
)

start = time.perf_counter()
cached_df.count()
print("Cache materialization:", time.perf_counter() - start)

start = time.perf_counter()
cached_df.count()
print("Cached reuse:", time.perf_counter() - start)

Cache materialization: 1.2552590419654734
Cached reuse: 0.2054154579527676


In [53]:
builtin_price_band = (
    order_items
    .withColumn(
        "price_band",
        F.when(F.col("price") < 50, "low")
         .when(F.col("price") < 150, "medium")
         .otherwise("high")
    )
)

builtin_price_band.explain("formatted")

== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:decimal(12,2),freight_value:decimal(12,2),item_total:decimal(13,2)>

(2) ColumnarToRow [codegen id : 1]
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]

(3) Project [codegen id : 1]
Output [9]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114, CASE WHEN (price#112 < 50.00) THEN low WHEN (price#112 < 150.

In [55]:
def classify_price(price):
    if price is None:
        return None
    elif price < 50:
        return "low"
    elif price < 150:
        return "medium"
    else:
        return "high"


classify_price_udf = udf(
    classify_price,
    StringType()
)

/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.13/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.13/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


In [56]:
udf_price_band = (
    order_items
    .withColumn(
        "price_band",
        classify_price_udf(F.col("price"))
    )
)

udf_price_band.explain("formatted")

== Physical Plan ==
* Project (4)
+- BatchEvalPython (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Batched: true
Location: InMemoryFileIndex [file:/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/data/silver/order_items]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:decimal(12,2),freight_value:decimal(12,2),item_total:decimal(13,2)>

(2) ColumnarToRow [codegen id : 1]
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]

(3) BatchEvalPython
Input [8]: [order_id#107, order_item_id#108, product_id#109, seller_id#110, shipping_limit_date#111, price#112, freight_value#113, item_total#114]
Arguments: [classify_price(price#112)#934]